### Part 3: Intelligence Layers

**Rubric coverage (25 pts)**
| Service | Pts | Key requirement |
|---|---|---|
| RAGService (LCEL) | 8 | Runnable chain, inline citations, evidence URLs |
| CAGService (Caching) | 8 | Two-tier cache, cosine ≥ 0.90, FAQ pre-warming |
| CRAGService (Correction) | 9 | Confidence scoring, corrective retrieval < 0.6 |

All thresholds and parameters are read from `config.yaml` via `context_engineering.config`.

### 1. Setup & Environment

In [ ]:
import sys
import os
import gc
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
src_path = project_root / "src"

for p in [str(project_root), str(src_path)]:
    if p not in sys.path:
        sys.path.insert(0, p)

load_dotenv(project_root / ".env")

print(f"📂 Project Root: {project_root}")
print("✅ Required packages are installed and ready.")

### 1.1 API Key Check & Config Imports

In [ ]:
keys = {
    "OpenAI":     os.getenv("OPENAI_API_KEY"),
    "OpenRouter": os.getenv("OPENROUTER_API_KEY"),
    "Google":     os.getenv("GEMINI_API_KEY"),
    "Groq":       os.getenv("GROQ_API_KEY"),
    "DeepSeek":   os.getenv("DEEPSEEK_API_KEY"),
    "Cohere":     os.getenv("COHERE_API_KEY"),
}
active_providers = [name for name, key in keys.items() if key]

if not active_providers:
    raise EnvironmentError("❌ No API keys found! Check your .env file.")

try:
    from context_engineering.config import (
        VECTOR_DIR, CACHE_DIR, TOP_K_RESULTS,
        CHAT_MODEL, EMBEDDING_MODEL, PROVIDER, LLM_TEMPERATURE,
        QDRANT_COLLECTIONS, QDRANT_DEFAULT_COLLECTION,
        CRAG_EXPANDED_K, CRAG_CONFIDENCE_THRESHOLD,
        CAG_SIMILARITY_THRESHOLD, CAG_HISTORY_TTL_HOURS,
    )
except ImportError as e:
    print(f"❌ Config import failed: {e}")
    print("Ensure context_engineering.config is reachable and config.yaml is populated.")
    raise

# Rubric guard: similarity threshold must be >= 0.90
assert CAG_SIMILARITY_THRESHOLD >= 0.90, (
    f"❌ CAG_SIMILARITY_THRESHOLD is {CAG_SIMILARITY_THRESHOLD} — rubric requires cosine ≥ 0.90. "
    "Update config.yaml."
)
# Rubric guard: confidence threshold must be 0.6
assert CRAG_CONFIDENCE_THRESHOLD == 0.6, (
    f"❌ CRAG_CONFIDENCE_THRESHOLD is {CRAG_CONFIDENCE_THRESHOLD} — rubric requires 0.6. "
    "Update config.yaml."
)

print(f"✅ Environment loaded — Provider: {PROVIDER} | Model: {CHAT_MODEL}")
print(f"🔑 Active keys          : {', '.join(active_providers)}")
print(f"🧠 Embeddings           : {EMBEDDING_MODEL}")
print(f"🎯 CAG similarity thresh: {CAG_SIMILARITY_THRESHOLD}  (rubric: ≥ 0.90)")
print(f"📊 CRAG confidence thresh: {CRAG_CONFIDENCE_THRESHOLD} (rubric: 0.6)")

### 1.2 Import Chat Services

In [ ]:
try:
    from context_engineering.application.chat_service import (
        RAGService, CAGService, CRAGService, CAGCache
    )
    print("✅ Chat services loaded: RAGService, CAGService, CRAGService, CAGCache")
except ImportError as e:
    print(f"❌ Failed to import chat services: {e}")
    print("Verify that context_engineering.application.chat_service exists and all classes are defined.")
    raise

### 2. RAGService — LCEL Chain with Inline Citations

Built on a **modern LCEL chain (Runnable)** — retriever → prompt → LLM → output parser —
with inline `[Source N]` citations and evidence URLs returned per response.

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from context_engineering.infrastructure.llm_providers import get_default_embeddings, get_chat_llm

if not VECTOR_DIR.exists():
    raise FileNotFoundError(
        f"❌ Vector DB not found at {VECTOR_DIR}. Run '02_the_chunking_lab.ipynb' first."
    )

try:
    embeddings = get_default_embeddings()
except Exception as e:
    raise RuntimeError(f"❌ Failed to load embeddings: {e}")

try:
    llm = get_chat_llm(temperature=LLM_TEMPERATURE)
except Exception as e:
    raise RuntimeError(f"❌ Failed to load chat LLM: {e}")

# Release stale Qdrant lock
if "client" in globals() and client is not None:
    try:
        client.close()
    except Exception:
        pass
    client = None
gc.collect()

lock_file = VECTOR_DIR / ".lock"
if lock_file.exists():
    try:
        lock_file.unlink()
        print("🔓 Removed stale lock file")
    except PermissionError:
        raise RuntimeError(
            "❌ Cannot remove lock — another process still holds it.\n"
            "   👉 Kernel → Shut Down All Kernels, then re-run."
        )

try:
    client = QdrantClient(path=str(VECTOR_DIR))
except Exception as e:
    raise RuntimeError(f"❌ Could not connect to Qdrant at {VECTOR_DIR}: {e}")

# Build one retriever per collection
retrievers = {}
for name in QDRANT_COLLECTIONS:
    try:
        retrievers[name] = QdrantVectorStore(
            client=client, collection_name=name, embedding=embeddings
        ).as_retriever(search_type="similarity", search_kwargs={"k": TOP_K_RESULTS})
    except Exception as e:
        print(f"  ⚠️  Could not create retriever for '{name}': {e}")

if QDRANT_DEFAULT_COLLECTION not in retrievers:
    raise RuntimeError(
        f"❌ Default collection '{QDRANT_DEFAULT_COLLECTION}' retriever could not be created. "
        "Ensure the collection is indexed in Part 2."
    )

retriever = retrievers[QDRANT_DEFAULT_COLLECTION]

print(f"✅ LLM: {CHAT_MODEL} | Embeddings: {EMBEDDING_MODEL}")
print(f"✅ Retrievers created: {list(retrievers.keys())}")
for name in QDRANT_COLLECTIONS:
    try:
        count = client.count(collection_name=name)
        print(f"   📊 {name:<30} {count.count:>5} vectors")
    except Exception as e:
        print(f"   ⚠️  {name} — could not count: {e}")

In [ ]:
rag_services = {}
for name in retrievers:
    try:
        rag_services[name] = RAGService(
            retriever=retrievers[name], llm=llm, k=TOP_K_RESULTS
        )
    except Exception as e:
        print(f"  ⚠️  Could not init RAGService for '{name}': {e}")

if QDRANT_DEFAULT_COLLECTION not in rag_services:
    raise RuntimeError(
        f"❌ Could not create RAGService for default collection '{QDRANT_DEFAULT_COLLECTION}'."
    )

rag_service = rag_services[QDRANT_DEFAULT_COLLECTION]

# Rubric guard: verify RAGService exposes a Runnable/LCEL chain attribute
chain_attr = next(
    (a for a in ["chain", "rag_chain", "pipeline", "runnable"]
     if hasattr(rag_service, a)),
    None,
)
if chain_attr:
    print(f"✅ LCEL chain attribute found: '{chain_attr}'")
else:
    print(
        "⚠️  WARNING: No LCEL chain attribute found on RAGService "
        "(looked for: chain, rag_chain, pipeline, runnable).\n"
        "   Rubric requires a modern LCEL Runnable chain — check RAGService implementation."
    )

print(f"✅ RAGService initialized for {len(rag_services)} collections")

### 2.1 RAGService — Test Across All Collections

In [ ]:
query = "What are the available land projects and payment plans?"
print(f"🔍 Query: {query}\n" + "=" * 80)

rag_collection_results = {}
for name, svc in rag_services.items():
    try:
        result = svc.generate(query)
        rag_collection_results[name] = result
        label = name.replace("primelands_", "")
        print(f"\n📦 {label}")
        print(f"   ⏱️  {result['generation_time']:.2f}s | 📄 {result['num_docs']} docs")
        print(f"   🤖 {result['answer'][:120]}...")
        for url in result.get("evidence_urls", [])[:2]:
            print(f"      📎 {url}")
    except Exception as e:
        print(f"\n  ❌ {name} — generate() failed: {e}")

# Rubric guard: at least the default collection must have produced a result
if QDRANT_DEFAULT_COLLECTION not in rag_collection_results:
    raise RuntimeError(
        f"❌ RAGService.generate() failed for default collection '{QDRANT_DEFAULT_COLLECTION}'."
    )

default_result = rag_collection_results[QDRANT_DEFAULT_COLLECTION]

# Rubric guard: inline citations — evidence_urls must be non-empty
evidence_urls = default_result.get("evidence_urls", [])
assert len(evidence_urls) > 0, (
    "❌ evidence_urls is empty — RAGService must return inline citation URLs per response. "
    "Check RAGService.generate() implementation."
)
print(f"\n✅ Evidence URLs present: {len(evidence_urls)} URL(s) returned.")

# Rubric guard: answer should contain citation markers e.g. [Source 1]
answer_text = default_result.get("answer", "")
has_citation_marker = "[Source" in answer_text or "[source" in answer_text or "[" in answer_text
if has_citation_marker:
    print("✅ Inline citation markers detected in answer.")
else:
    print(
        "⚠️  WARNING: No inline citation markers (e.g. [Source 1]) found in answer. "
        "Rubric requires inline citations — check the RAGService prompt template."
    )

### 3. CAGService — Two-Tier Semantic Cache (FAQs + History)

- **FAQ tier**: Pre-warmed from `config/faqs.yaml`, never expires
- **History tier**: User queries cached with TTL from `config.yaml` (`cag.history_ttl_hours`)
- **Matching**: Cosine similarity ≥ `cag.similarity_threshold` (0.90) — catches paraphrases
- **Hit tracking**: Running totals of hits, misses, and hit rate reported after each test

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)

try:
    cache = CAGCache(
        cache_dir=CACHE_DIR,
        embedder=embeddings,
        similarity_threshold=CAG_SIMILARITY_THRESHOLD,
        history_ttl_hours=CAG_HISTORY_TTL_HOURS,
    )
    cag_service = CAGService(rag_service=rag_service, cache=cache)
except Exception as e:
    raise RuntimeError(f"❌ Failed to initialise CAGService: {e}")

try:
    stats = cache.stats()
except Exception as e:
    raise RuntimeError(f"❌ cache.stats() failed: {e}")

print("✅ CAGService initialized")
print(f"📁 Cache directory      : {CACHE_DIR}")
print(f"📊 Cached responses     : {stats['total_cached']}")
print(f"🎯 Similarity threshold : {stats['similarity_threshold']}")
print(f"⏰ History TTL          : {stats['history_ttl_hours']}h")
print(f"💾 Cache size           : {stats.get('cache_size_kb', 0):.2f} KB")

### 3.1 FAQ Pre-Warming

In [ ]:
try:
    from context_engineering.config import KNOWN_FAQS
except ImportError as e:
    raise ImportError(f"❌ Could not import KNOWN_FAQS from config: {e}")

if not KNOWN_FAQS:
    print("⚠️  WARNING: KNOWN_FAQS is empty — FAQ tier will have nothing to pre-warm.")
    print("   Populate config/faqs.yaml to enable this feature.")
else:
    try:
        loaded = cag_service.load_faqs(KNOWN_FAQS)
        print(f"✅ Loaded {loaded} FAQs from config/faqs.yaml")
    except Exception as e:
        raise RuntimeError(f"❌ load_faqs() failed: {e}")

In [ ]:
try:
    start = time.time()
    cag_service.warm_faqs(verbose=False)
    duration = time.time() - start
except Exception as e:
    raise RuntimeError(f"❌ warm_faqs() failed: {e}")

stats = cag_service.cache.stats()
warmed_count = stats['total_cached']

print(f"✅ Cache warmed in {duration:.2f}s — Total: {warmed_count} | Size: {stats.get('cache_size_kb', 0):.2f} KB")

# Rubric guard: FAQ warm must have populated the cache
assert warmed_count > 0, (
    "❌ Cache is still empty after warm_faqs() — check load_faqs() and warm_faqs() implementation."
)

### 3.2 Cache Hit / Miss Demonstration

Tests the FAQ tier (must respond in < 500ms) and History tier (cold then warm).
Hit rate is tracked across all calls.

In [ ]:
cag_hit_log = []

print("=" * 60)
print("CACHE HIT / MISS DEMONSTRATION")
print("=" * 60)

# ── FAQ Tier ──────────────────────────────────────────────────
faq_query = KNOWN_FAQS[0] if KNOWN_FAQS else "What payment plans are available?"
try:
    faq_r = cag_service.generate(faq_query, verbose=False)
except Exception as e:
    raise RuntimeError(f"❌ CAGService.generate() failed on FAQ query: {e}")

faq_ms = faq_r["generation_time"] * 1000
status = "✅ PASS" if faq_ms < 500 else "⚠️  SLOW"
cag_hit_log.append(faq_r["cache_hit"])

print(f"\n📋 FAQ TIER")
print(f"   Query  : '{faq_query}'")
print(f"   Time   : {faq_ms:.1f}ms {status} | Hit: {faq_r['cache_hit']} | Source: {faq_r['cache_source'] or 'RAG Pipeline'}")

# Rubric quick check: FAQ query must respond in < 500ms
assert faq_ms < 500, (
    f"❌ FAQ tier took {faq_ms:.1f}ms — rubric requires < 500ms. "
    "Ensure the FAQ cache is pre-warmed before this query."
)

# Rubric guard: FAQ query should be a cache hit
if not faq_r["cache_hit"]:
    print(
        "⚠️  WARNING: FAQ query was a cache MISS. "
        "Verify that warm_faqs() correctly indexed this FAQ."
    )

# ── History Tier ───────────────────────────────────────────────
test_query = "What is the price of land in Horana?"
print(f"\n🕐 HISTORY TIER")
print(f"   Query  : '{test_query}'")

try:
    r1 = cag_service.generate(test_query, verbose=False)    # cold
except Exception as e:
    raise RuntimeError(f"❌ Cold history query failed: {e}")
cag_hit_log.append(r1["cache_hit"])
print(f"   Cold   : {r1['generation_time']:.4f}s | Hit: {r1['cache_hit']} | Source: {r1['cache_source'] or 'RAG Pipeline'}")

try:
    r2 = cag_service.generate(test_query, verbose=False)    # warm
except Exception as e:
    raise RuntimeError(f"❌ Warm history query failed: {e}")
cag_hit_log.append(r2["cache_hit"])
print(f"   Warm   : {r2['generation_time']:.4f}s | Hit: {r2['cache_hit']} | Source: {r2['cache_source'] or 'RAG Pipeline'}")

# Rubric guard: warm query must be a cache hit
assert r2["cache_hit"], (
    "❌ Warm history query was not a cache HIT — the history tier is not caching responses. "
    "Check CAGService/CAGCache implementation."
)

speedup = r1["generation_time"] / r2["generation_time"] if r2["generation_time"] > 0 else float("inf")
print(f"   Speedup (cold→warm): {speedup:.1f}x")

if speedup < 2:
    print("   ⚠️  WARNING: Speedup < 2x — cache may not be providing meaningful latency savings.")

# ── Cache Statistics Summary ───────────────────────────────────
stats = cag_service.cache.stats()
total_calls = len(cag_hit_log)
total_hits  = sum(cag_hit_log)
hit_rate    = total_hits / total_calls * 100 if total_calls else 0

print(f"\n{'=' * 60}")
print("CACHE STATISTICS")
print(f"{'=' * 60}")
print(f"   Total calls         : {total_calls}")
print(f"   Cache hits          : {total_hits}")
print(f"   Cache misses        : {total_calls - total_hits}")
print(f"   Hit rate            : {hit_rate:.1f}%")
print(f"   Speedup (cold→warm) : {speedup:.1f}x")
print(f"   Total cached        : {stats['total_cached']}")
print(f"   Cache size          : {stats.get('cache_size_kb', 0):.2f} KB")

# Rubric guard: cache must be tracking hits
assert total_hits > 0, (
    "❌ No cache hits recorded across all test queries — "
    "two-tier caching is not functioning. Inspect CAGCache."
)

### 4. CRAGService — Corrective RAG with Confidence Scoring

- **Confidence threshold**: `CRAG_CONFIDENCE_THRESHOLD` from config (0.6) — below this,
  corrective retrieval triggers
- **Initial retrieval**: `TOP_K_RESULTS` docs
- **Corrective retrieval**: `CRAG_EXPANDED_K` docs (wider search when confidence is low)
- **Test design**: Off-domain query (low confidence → correction fires) + in-domain query

In [ ]:
try:
    crag_service = CRAGService(
        retriever=retriever,
        llm=llm,
        initial_k=TOP_K_RESULTS,
        expanded_k=CRAG_EXPANDED_K,
    )
except Exception as e:
    raise RuntimeError(f"❌ Failed to initialise CRAGService: {e}")

print(f"✅ CRAGService initialized")
print(f"   Initial k   : {TOP_K_RESULTS}")
print(f"   Corrective k: {CRAG_EXPANDED_K}")
print(f"   Threshold   : {CRAG_CONFIDENCE_THRESHOLD}")

# Rubric guard: expanded_k must be strictly greater than initial_k
assert CRAG_EXPANDED_K > TOP_K_RESULTS, (
    f"❌ CRAG_EXPANDED_K ({CRAG_EXPANDED_K}) must be > TOP_K_RESULTS ({TOP_K_RESULTS}). "
    "Corrective retrieval must fetch more documents than the initial pass."
)

### 4.1 Corrective Retrieval — Trigger Demonstration

**Test 1** — off-domain query (medical topic) → confidence < `CRAG_CONFIDENCE_THRESHOLD`
→ corrective retrieval fires with expanded k.

**Test 2** — off-domain query (ML/tech) → confidence < threshold → correction fires again.

**Test 3** — specific in-domain query → confidence ≥ threshold → no correction needed.

In [ ]:
test_cases = [
    {
        "query": "What are the symptoms and treatment options for Type 2 diabetes?",
        "label": "Off-domain — medical topic, expect correction (confidence < 0.6)",
        "expect_correction": True,
    },
    {
        "query": "Explain how transformer neural networks work in deep learning.",
        "label": "Off-domain — ML/tech topic, expect correction (confidence < 0.6)",
        "expect_correction": True,
    },
    {
        "query": "What are the available land projects in Colombo?",
        "label": "In-domain — expect no correction (confidence ≥ 0.6)",
        "expect_correction": False,
    },
]

print("=" * 70)
print("CORRECTIVE RAG (CRAG) TEST")
print("=" * 70)

crag_results = []
for i, test in enumerate(test_cases, 1):
    print(f"\nTest {i}: {test['label']}")
    print("-" * 70)
    try:
        result = crag_service.generate(
            test["query"],
            confidence_threshold=CRAG_CONFIDENCE_THRESHOLD,
            verbose=False,
        )
    except Exception as e:
        print(f"   ❌ generate() failed: {e}")
        crag_results.append(None)
        continue

    crag_results.append(result)
    triggered = "🔧 CORRECTIVE TRIGGERED" if result["correction_applied"] else "✅ Standard (conf OK)"
    print(f"   Confidence  : {result['confidence_initial']:.2f} → {result['confidence_final']:.2f}")
    print(f"   Docs used   : {result['docs_used']} | Time: {result['generation_time']:.2f}s")
    print(f"   Correction  : {triggered}")
    print(f"   Answer      : {result['answer'][:250]}...")

    # Per-test rubric guard: check correction behaviour matches expectation
    if test["expect_correction"] and not result["correction_applied"]:
        print(
            f"   ⚠️  WARNING: Expected corrective retrieval to trigger for off-domain query, "
            f"but confidence_initial={result['confidence_initial']:.2f} was above threshold {CRAG_CONFIDENCE_THRESHOLD}."
        )
    elif not test["expect_correction"] and result["correction_applied"]:
        print(
            f"   ⚠️  WARNING: In-domain query triggered corrective retrieval unexpectedly. "
            f"confidence_initial={result['confidence_initial']:.2f} was below threshold {CRAG_CONFIDENCE_THRESHOLD}."
        )

### 4.2 Correction Impact Summary

In [ ]:
print("=" * 70)
print("CORRECTION IMPACT")
print("=" * 70)

valid_results = [(t, r) for t, r in zip(test_cases, crag_results) if r is not None]

for i, (test, result) in enumerate(valid_results, 1):
    delta      = result["confidence_final"] - result["confidence_initial"]
    trend      = "⬆️ improved" if delta > 0.01 else ("➡️ unchanged" if abs(delta) <= 0.01 else "⬇️ declined")
    docs_delta = result["docs_used"] - TOP_K_RESULTS
    extra      = f" (+{docs_delta} extra docs)" if docs_delta > 0 else ""
    print(
        f"   Test {i}: {result['confidence_initial']:.2f} → {result['confidence_final']:.2f} "
        f"{trend}{extra} | corrected={result['correction_applied']}"
    )

# Rubric guard: corrective retrieval must have triggered at least once
corrected_results = [r for r in crag_results if r is not None and r["correction_applied"]]
assert len(corrected_results) > 0, (
    "❌ Corrective retrieval never triggered across any test query! "
    "Use a more off-domain query or lower CRAG_CONFIDENCE_THRESHOLD to demonstrate the feature."
)

# Rubric guard: when correction triggers, it must use more docs than the initial k
for r in corrected_results:
    assert r["docs_used"] >= TOP_K_RESULTS, (
        f"❌ Corrected query used {r['docs_used']} docs but TOP_K_RESULTS={TOP_K_RESULTS}. "
        "Corrective retrieval must expand the document set."
    )

# Rubric guard: correction should demonstrate improvement (confidence_final >= confidence_initial)
degraded = [r for r in corrected_results if r["confidence_final"] < r["confidence_initial"] - 0.05]
if degraded:
    print(
        f"\n  ⚠️  WARNING: {len(degraded)} corrected query(s) show confidence degradation after correction. "
        "Rubric requires that corrective retrieval demonstrates improvement."
    )
else:
    print("\n  ✅ Confidence maintained or improved after corrective retrieval.")

print("\n✅ Corrective retrieval demonstrated — triggers and behaviour confirmed.")

### 5. RAG vs CAG vs CRAG — Side-by-Side Comparison

In [ ]:
comparison_query = "What are the available land projects and payment plans?"
print(f"🔍 {comparison_query}\n")

comparison_data = []

for name, svc in rag_services.items():
    try:
        r = svc.generate(comparison_query)
        comparison_data.append({
            "Strategy":        name.replace("primelands_", ""),
            "Service":         "RAG",
            "Latency (s)":     f"{r['generation_time']:.2f}",
            "Docs Retrieved":  r["num_docs"],
            "Cache Used":      "No",
            "Self-Correcting": "No",
            "Best For":        "General queries",
        })
    except Exception as e:
        print(f"  ⚠️  RAG generate() failed for '{name}': {e}")

try:
    cag_result = cag_service.generate(comparison_query, use_cache=True, verbose=False)
    comparison_data.append({
        "Strategy":        f"{QDRANT_DEFAULT_COLLECTION.replace('primelands_', '')} (CAG)",
        "Service":         "CAG",
        "Latency (s)":     f"{cag_result['generation_time']:.2f}",
        "Docs Retrieved":  "Cached" if cag_result["cache_hit"] else TOP_K_RESULTS,
        "Cache Used":      "Yes" if cag_result["cache_hit"] else "No",
        "Self-Correcting": "No",
        "Best For":        "Frequent/FAQ queries",
    })
except Exception as e:
    print(f"  ⚠️  CAG generate() failed: {e}")
    cag_result = None

try:
    crag_result = crag_service.generate(
        comparison_query,
        confidence_threshold=CRAG_CONFIDENCE_THRESHOLD,
        verbose=False,
    )
    comparison_data.append({
        "Strategy":        f"{QDRANT_DEFAULT_COLLECTION.replace('primelands_', '')} (CRAG)",
        "Service":         "CRAG",
        "Latency (s)":     f"{crag_result['generation_time']:.2f}",
        "Docs Retrieved":  crag_result["docs_used"],
        "Cache Used":      "No",
        "Self-Correcting": "Yes" if crag_result["correction_applied"] else "No (conf OK)",
        "Best For":        "Complex/uncertain queries",
    })
except Exception as e:
    print(f"  ⚠️  CRAG generate() failed: {e}")
    crag_result = None

if not comparison_data:
    raise RuntimeError("❌ No results produced for the side-by-side comparison.")

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

# Highlight fastest RAG strategy
rag_rows = df[df["Service"] == "RAG"].copy()
if not rag_rows.empty:
    rag_rows["_lat"] = rag_rows["Latency (s)"].astype(float)
    fastest = rag_rows.loc[rag_rows["_lat"].idxmin(), "Strategy"]
    print(f"\n⚡ Fastest RAG strategy: {fastest}")

# Rubric check: all three service types must be represented
services_present = set(df["Service"].tolist())
for required_svc in ["RAG", "CAG", "CRAG"]:
    if required_svc not in services_present:
        print(f"  ⚠️  WARNING: '{required_svc}' is missing from the comparison table.")
    else:
        print(f"  ✅ {required_svc} present in comparison.")

### 6. Interactive — Ask Your Own Question

In [ ]:
# Change this to any question — avoids blocking input() on automated runs
YOUR_QUESTION = "What payment plans are available for buying land?"

print(f"\n🔍 '{YOUR_QUESTION}'\n" + "=" * 80)

# ── Standard RAG ──────────────────────────────────────────────
print("\n1️⃣  Standard RAG")
try:
    rag_r = rag_service.generate(YOUR_QUESTION)
    print(f"⏱️  {rag_r['generation_time']:.2f}s | 📄 {rag_r['num_docs']} docs")
    print(rag_r["answer"])
    for url in rag_r.get("evidence_urls", [])[:3]:
        print(f"  📎 {url}")
except Exception as e:
    print(f"  ❌ RAG failed: {e}")
    rag_r = None

# ── Cache-Augmented Generation ─────────────────────────────────
print("\n2️⃣  Cache-Augmented Generation (CAG)")
try:
    cag_r = cag_service.generate(YOUR_QUESTION, use_cache=True, verbose=False)
    source_label = f"HIT ({cag_r['cache_source'].upper()})" if cag_r["cache_hit"] else "MISS"
    print(f"⏱️  {cag_r['generation_time']:.2f}s | 💾 {source_label}")
    print(cag_r["answer"])
except Exception as e:
    print(f"  ❌ CAG failed: {e}")
    cag_r = None

# ── Corrective RAG ─────────────────────────────────────────────
print("\n3️⃣  Corrective RAG (CRAG)")
try:
    crag_r = crag_service.generate(
        YOUR_QUESTION,
        confidence_threshold=CRAG_CONFIDENCE_THRESHOLD,
        verbose=False,
    )
    action = "corrective" if crag_r["correction_applied"] else "standard"
    print(f"⏱️  {crag_r['generation_time']:.2f}s | 📊 Confidence: {crag_r['confidence_initial']:.2f} → {crag_r['confidence_final']:.2f} | 🔧 {action}")
    print(crag_r["answer"])
except Exception as e:
    print(f"  ❌ CRAG failed: {e}")
    crag_r = None

# ── Summary Table ──────────────────────────────────────────────
print("\n" + "=" * 80)
print(f"{'System':<15} {'Time (s)':<12} {'Feature'}")
print("-" * 50)
if rag_r:
    print(f"{'Standard RAG':<15} {rag_r['generation_time']:<12.2f} Baseline")
if cag_r:
    source_label = f"HIT ({cag_r['cache_source'].upper()})" if cag_r["cache_hit"] else "MISS"
    print(f"{'CAG':<15} {cag_r['generation_time']:<12.2f} Cache: {source_label}")
if crag_r:
    action = "corrective" if crag_r["correction_applied"] else "standard"
    print(f"{'CRAG':<15} {crag_r['generation_time']:<12.2f} Confidence: {crag_r['confidence_initial']:.2f} | {action}")